[Guide Index](README.md)


# 03a SPARQL Entailment

`StarLayerGraph.query()` lets a query itself specify an *entailment regime* to apply via `entailment=`, without first running `infer()` on the underlying graph. Every entailment regime - what each one means, and a worked example of both `infer()` and `query(entailment=...)` forms - is explained once, in full, in the [Graph Inferencing guide](02b-graphs-inferencing.ipynb). This guide covers only what's specific to the query-time form: how caching works (`infer=`), and how to detect an inconsistency via `ASK`.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier sections.


In [1]:
from starlayer.graph import ENTAILMENT, StarLayerGraph, Namespace, RDF

EX = Namespace("http://example.org/")

## 1. Applying entailment at query time

`query(sparql, entailment=...)` accepts the same `ENTAILMENT.*` vocabulary `infer()` does. `ENTAILMENT.RDF`/`ENTAILMENT.RDFS` are handled via a query-time rewrite of the query itself - no copy of the graph's data is made, and the graph's own triple count never changes. `ENTAILMENT['OWL-RDF-Based']` instead computes a closure once (see section 2 below for how that result is cached/invalidated across subsequent queries).

In [2]:
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Manager rdfs:subClassOf ex:Employee .
    ex:alice a ex:Manager .
""", format="turtle12")

QUERY = "PREFIX ex: <http://example.org/> SELECT ?x WHERE { ?x a ex:Employee }"

print("plain (no entailment):     ", [str(r.x) for r in g.query(QUERY)])
print("entailment=ENTAILMENT.RDFS:", [str(r.x) for r in g.query(QUERY, entailment=ENTAILMENT.RDFS)])
print("triple count unchanged - no data was copied or added:", len(g))

plain (no entailment):      []
entailment=ENTAILMENT.RDFS: ['http://example.org/alice']
triple count unchanged - no data was copied or added: 2


## 2. Caching via `infer=`

Unlike `entailment=ENTAILMENT.RDFS`, `entailment=ENTAILMENT['OWL-RDF-Based']` requires a potentially time-consuming calculation of the entailed graph.  

By default (`infer="changed"`), `entailment=ENTAILMENT['OWL-RDF-Based']` will cache the entailed graph and reuse it for subsequent queries unless the underlying graph changes.  The other choices are `infer="cached"` which will force the re-use of the cached entailed graph regardless of changes and  `infer="always"` will force a re-compute of the entailed graph.  

In [3]:
# alice is a manager, but she is also a teamlead, because manager and teamlead are the same
g_owl = StarLayerGraph()
g_owl.bind("ex", EX)
g_owl.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:Manager owl:equivalentClass ex:TeamLead .
    ex:alice a ex:Manager .
""", format="turtle12")

QUERY_OWL = "PREFIX ex: <http://example.org/> SELECT ?x WHERE { ?x a ex:TeamLead }"
print("before:", sorted(str(r.x) for r in g_owl.query(QUERY_OWL, entailment=ENTAILMENT['OWL-RDF-Based'])))

g_owl.add((EX.carol, RDF.type, EX.Manager))
rows = g_owl.query(QUERY_OWL, entailment=ENTAILMENT['OWL-RDF-Based'])
print("default (changed), picks up carol:", sorted(str(r.x) for r in rows))

g_owl.add((EX.dave, RDF.type, EX.Manager))
stale = g_owl.query(QUERY_OWL, entailment=ENTAILMENT['OWL-RDF-Based'], infer="cached")
print("infer=\"cached\" (misses dave - stale on purpose):", sorted(str(r.x) for r in stale))

fresh = g_owl.query(QUERY_OWL, entailment=ENTAILMENT['OWL-RDF-Based'], infer="always")
print("infer=\"always\" (picks up dave):                 ", sorted(str(r.x) for r in fresh))

before: ['http://example.org/alice']
default (changed), picks up carol: ['http://example.org/alice', 'http://example.org/carol']
infer="cached" (misses dave - stale on purpose): ['http://example.org/alice', 'http://example.org/carol']
infer="always" (picks up dave):                  ['http://example.org/alice', 'http://example.org/carol', 'http://example.org/dave']


## 3. Detecting inconsistencies via `ASK`

`entailment=ENTAILMENT['OWL-RDF-Based']` inherits `owlrl`'s consistency checks (see the Inferencing guide's [section 4.5](02b-graphs-inferencing.ipynb#4.5-Detecting-inconsistencies) for what an inconsistency means and how it's flagged - an `err:error` triple, not an exception). When querying using `OWL-RDF-Based` entailment, a result will be returned even if the underlying inference detects an inconsistency. Running a simple `ASK` query can detect the inconsistency:

In [4]:
ASK_INCONSISTENT = """
PREFIX err: <http://www.daml.org/2002/03/agents/agent-ont#>
ASK { ?e a err:ErrorMessage }
"""

# Dog and Cat are declared disjoint, but fido is asserted to be both
g_bad = StarLayerGraph()
g_bad.bind("ex", EX)
g_bad.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:Dog owl:disjointWith ex:Cat .
    ex:fido a ex:Dog, ex:Cat .
''', format="turtle12")

print("g_bad (genuine inconsistency):  ", bool(g_bad.query(ASK_INCONSISTENT, entailment=ENTAILMENT['OWL-RDF-Based'])))

# a normal query still returns a result - nothing is retracted, the inconsistency isn't surfaced here
rows = g_bad.query("PREFIX ex: <http://example.org/> SELECT ?c WHERE { ex:fido a ?c }", entailment=ENTAILMENT['OWL-RDF-Based'])
print("fido's classes (still returned despite the inconsistency):", sorted(str(r.c) for r in rows))

g_bad (genuine inconsistency):   True
fido's classes (still returned despite the inconsistency): ['http://example.org/Cat', 'http://example.org/Dog']
